# CSRNet — Inference Notebook
### Mashrur
---

## Section 1 — Setup

In [ ]:
# A1 — Install dependencies & clone repo
!pip install -q kagglehub tensorboardX gradio
!git clone https://github.com/CommissarMa/CSRNet-pytorch.git
print('Done!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 4.9 MB/s eta 0:00:00
Cloning into 'CSRNet-pytorch'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 53 (delta 10), reused 6 (delta 6), pack-reused 34 (from 1)
Receiving objects: 100% (53/53), 19.86 KiB | 9.93 MiB/s, done.
Resolving deltas: 100% (18/18), done.
Done!


In [ ]:
# A2 — Download dataset (used for example images in demo)
import kagglehub
path = kagglehub.dataset_download('tthien/shanghaitech')
print('Dataset path:', path)

100%|██████████| 333M/333M [00:03<00:00, 93.4MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/tthien/shanghaitech/versions/1


In [ ]:
# A3 — Download pretrained weights (epoch 73, MAE ~9.4)
import gdown
gdown.download(
    'https://drive.google.com/uc?id=1tV51fSwKoago3fbwowlXlWDnJsHcgyrY',
    '/content/73.pth',
    quiet=False
)
print('Weights downloaded!')

Downloading...
From (original): https://drive.google.com/uc?id=1tV51fSwKoago3fbwowlXlWDnJsHcgyrY
From (redirected): https://drive.google.com/uc?id=1tV51fSwKoago3fbwowlXlWDnJsHcgyrY&confirm=t&uuid=aee14e56-fd5d-48a7-a39b-032846c9128d
To: /content/73.pth
100%|██████████| 65.1M/65.1M [00:01<00:00, 53.6MB/s]

Weights downloaded!


In [ ]:
# A4 — Load model
import sys
sys.path.insert(0, '/content/CSRNet-pytorch')

import torch
from model import CSRNet

model = CSRNet().cuda()
model.load_state_dict(torch.load('/content/73.pth'))
model.eval()
print('Model loaded! MAE ~9.4 on ShanghaiTech Part B')

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 177MB/s]


Model loaded! MAE ~9.4 on ShanghaiTech Part B


## Section 2 — Video Inference & Gradio Demo

In [ ]:
import cv2
import torch
import numpy as np
import gradio as gr
import pandas as pd
import tempfile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image

# ---------------- DEVICE ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- LOAD MODEL ----------------
model = CSRNet().to(device)
model.load_state_dict(torch.load("73.pth", map_location=device))
model.eval()

# ---------------- TRANSFORM ----------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ---------------- VIDEO PROCESS FUNCTION ----------------
def process_video(video_path, threshold):
    cap = cv2.VideoCapture(video_path)

    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 24
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    output_path = "output.mp4"
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_id = 0
    frame_counts = []
    alert_text = "NORMAL"

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_id += 1

        # ---- Skip frames (performance) ----
        if frame_id % 2 != 0:
            out.write(frame)
            continue

        # ---- Preprocess ----
        img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        input_tensor = transform(img).unsqueeze(0).to(device)

        # ---- Inference ----
        with torch.no_grad():
            density_map = model(input_tensor)

        density_map = density_map.squeeze().cpu().numpy()
        count = density_map.sum()
        frame_counts.append({"frame": frame_id, "count": int(count)})

        # ---- Normalize density map ----
        norm_map = cv2.normalize(density_map, None, 0, 255, cv2.NORM_MINMAX)
        norm_map = norm_map.astype(np.uint8)

        # ---- Resize to match frame ----
        norm_map = cv2.resize(norm_map, (frame.shape[1], frame.shape[0]))

        # ---- Threshold to get "crowd regions" ----
        _, thresh = cv2.threshold(norm_map, 80, 255, cv2.THRESH_BINARY)

        # ---- Find contours ----
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        overlay = frame.copy()

        # ---- Draw green outlines ----
        for cnt in contours:
            if cv2.contourArea(cnt) > 200:
                cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)

        # ---- Alert ----
        alert_text = "NORMAL"
        color = (0, 255, 0)

        if count > threshold:
            alert_text = "ALERT: CROWD LIMIT EXCEEDED"
            color = (0, 0, 255)

        # ---- Draw text ----
        cv2.putText(overlay, f"Count: {int(count)}",
                    (20, 40), cv2.FONT_HERSHEY_SIMPLEX,
                    1, color, 2)
        cv2.putText(overlay, alert_text,
                    (20, 80), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, color, 2)

        out.write(overlay)

    cap.release()
    out.release()

    # ---- Build CSV ----
    df = pd.DataFrame(frame_counts)
    csv_path = tempfile.mktemp(suffix=".csv")
    df.to_csv(csv_path, index=False)

    # ---- Build plot ----
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(df["frame"], df["count"], color="royalblue", linewidth=1.5)
    ax.fill_between(df["frame"], df["count"], alpha=0.2, color="royalblue")
    ax.axhline(y=threshold, color="red", linestyle="--", label=f"Threshold ({int(threshold)})")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Count")
    ax.set_title("Crowd Count Over Time")
    ax.legend()
    plot_path = tempfile.mktemp(suffix=".png")
    plt.savefig(plot_path, bbox_inches="tight")
    plt.close()

    peak = df["count"].max()
    avg = round(df["count"].mean(), 1)
    summary = f"Peak: {peak} | Avg: {avg} | Final: {alert_text}"

    return output_path, plot_path, csv_path, summary


# ---------------- GRADIO UI ----------------
demo = gr.Interface(
    fn=process_video,
    inputs=[
        gr.Video(label="Upload Crowd Video"),
        gr.Slider(minimum=10, maximum=500, value=50, step=10, label="Crowd Threshold")
    ],
    outputs=[
        gr.Video(label="Processed Video"),
        gr.Image(label="Count Over Time"),
        gr.File(label="Download CSV Report"),
        gr.Text(label="Summary")
    ],
    title="Crowd Monitoring System",
    description="Upload a video, set your crowd limit — get annotated output, analytics graph, and a CSV report."
)

demo.launch()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e4adaaef90f23e7a09.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
